## Dealing with Unix Timestamp in Spark Data Frames

1. Unix Timestamp is the integer value starting from January 1, 1970 UTC
2. The starting time is known as epoch and is incremented by 1 second
3. unix_timestamp()
4. fromunix()
5. col(unixtime).cast('DATE') is not supported, rather  col(unixtime).cast('TIMESTAMP') works fine

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Dealing with Unix Timestamp in Spark Data Frames").getOrCreate()

24/03/18 21:45:18 WARN Utils: Your hostname, Qasim resolves to a loopback address: 127.0.1.1; using 172.30.54.131 instead (on interface eth0)
24/03/18 21:45:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/03/18 21:45:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/03/18 21:45:20 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/03/18 21:45:20 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
24/03/18 21:45:20 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


In [2]:
list = [
    (20231026, "2023-10-26", "2023-10-26 15:30:00"),
    (20240201, "2024-02-01", "2024-02-01 10:10:10"),
    (20231231, "2023-12-31", "2023-12-31 23:59:59"),
    (20240315, "2024-03-15", "2024-03-15 09:00:00")
]

list

[(20231026, '2023-10-26', '2023-10-26 15:30:00'),
 (20240201, '2024-02-01', '2024-02-01 10:10:10'),
 (20231231, '2023-12-31', '2023-12-31 23:59:59'),
 (20240315, '2024-03-15', '2024-03-15 09:00:00')]

In [3]:
df = spark.createDataFrame(list, schema='''
        dateid INTEGER, date STRING, timestamp STRING
    ''')

In [4]:
df.show()

+--------+----------+-------------------+
|  dateid|      date|          timestamp|
+--------+----------+-------------------+
|20231026|2023-10-26|2023-10-26 15:30:00|
|20240201|2024-02-01|2024-02-01 10:10:10|
|20231231|2023-12-31|2023-12-31 23:59:59|
|20240315|2024-03-15|2024-03-15 09:00:00|
+--------+----------+-------------------+



In [7]:
from pyspark.sql.functions import unix_timestamp, from_unixtime, col

In [22]:
help(unix_timestamp)

Help on function unix_timestamp in module pyspark.sql.functions:

unix_timestamp(timestamp: Union[ForwardRef('ColumnOrName'), NoneType] = None, format: str = 'yyyy-MM-dd HH:mm:ss') -> pyspark.sql.column.Column
    Convert time string with given pattern ('yyyy-MM-dd HH:mm:ss', by default)
    to Unix time stamp (in seconds), using the default timezone and the default
    locale, returns null if failed.
    
    if `timestamp` is None, then it returns current timestamp.
    
    .. versionadded:: 1.5.0
    
    .. versionchanged:: 3.4.0
        Supports Spark Connect.
    
    Parameters
    ----------
    timestamp : :class:`~pyspark.sql.Column` or str, optional
        timestamps of string values.
    format : str, optional
        alternative format to use for converting (default: yyyy-MM-dd HH:mm:ss).
    
    Returns
    -------
    :class:`~pyspark.sql.Column`
        unix time as long integer.
    
    Examples
    --------
    >>> spark.conf.set("spark.sql.session.timeZone", "Ame

In [13]:
df.select('*') \
    .withColumn('date', unix_timestamp('date', 'yyyy-MM-dd')) \
    .withColumn('timestamp', unix_timestamp('timestamp')) \
    .show()

+--------+----------+----------+
|  dateid|      date| timestamp|
+--------+----------+----------+
|20231026|1698264000|1698319800|
|20240201|1706731200|1706767810|
|20231231|1703966400|1704052799|
|20240315|1710446400|1710478800|
+--------+----------+----------+



In [20]:
df.select('*') \
    .withColumn('date_id_formatted', unix_timestamp(col('dateid').cast('STRING'), 'yyyy-MM-dd')) \
    .withColumn('date_formatted', unix_timestamp('date', 'yyyy-MM-dd')) \
    .withColumn('timestamp_formatted', unix_timestamp('timestamp')) \
    .show()

+--------+----------+-------------------+-----------------+--------------+-------------------+
|  dateid|      date|          timestamp|date_id_formatted|date_formatted|timestamp_formatted|
+--------+----------+-------------------+-----------------+--------------+-------------------+
|20231026|2023-10-26|2023-10-26 15:30:00|             NULL|    1698264000|         1698319800|
|20240201|2024-02-01|2024-02-01 10:10:10|             NULL|    1706731200|         1706767810|
|20231231|2023-12-31|2023-12-31 23:59:59|             NULL|    1703966400|         1704052799|
|20240315|2024-03-15|2024-03-15 09:00:00|             NULL|    1710446400|         1710478800|
+--------+----------+-------------------+-----------------+--------------+-------------------+



In [21]:
df.select('*') \
    .withColumn('date_id_formatted', unix_timestamp(col('dateid').cast('STRING'), 'yyyyMMdd')) \
    .withColumn('date_formatted', unix_timestamp('date', 'yyyy-MM-dd')) \
    .withColumn('timestamp_formatted', unix_timestamp('timestamp')) \
    .show()

+--------+----------+-------------------+-----------------+--------------+-------------------+
|  dateid|      date|          timestamp|date_id_formatted|date_formatted|timestamp_formatted|
+--------+----------+-------------------+-----------------+--------------+-------------------+
|20231026|2023-10-26|2023-10-26 15:30:00|       1698264000|    1698264000|         1698319800|
|20240201|2024-02-01|2024-02-01 10:10:10|       1706731200|    1706731200|         1706767810|
|20231231|2023-12-31|2023-12-31 23:59:59|       1703966400|    1703966400|         1704052799|
|20240315|2024-03-15|2024-03-15 09:00:00|       1710446400|    1710446400|         1710478800|
+--------+----------+-------------------+-----------------+--------------+-------------------+



In [15]:
df.withColumn('date_formatted', unix_timestamp('date', 'yyyy-MM-dd')) \
    .withColumn('timestamp_formatted', unix_timestamp('timestamp')) \
    .show()

+--------+----------+-------------------+--------------+-------------------+
|  dateid|      date|          timestamp|date_formatted|timestamp_formatted|
+--------+----------+-------------------+--------------+-------------------+
|20231026|2023-10-26|2023-10-26 15:30:00|    1698264000|         1698319800|
|20240201|2024-02-01|2024-02-01 10:10:10|    1706731200|         1706767810|
|20231231|2023-12-31|2023-12-31 23:59:59|    1703966400|         1704052799|
|20240315|2024-03-15|2024-03-15 09:00:00|    1710446400|         1710478800|
+--------+----------+-------------------+--------------+-------------------+



In [23]:
help(from_unixtime)

Help on function from_unixtime in module pyspark.sql.functions:

from_unixtime(timestamp: 'ColumnOrName', format: str = 'yyyy-MM-dd HH:mm:ss') -> pyspark.sql.column.Column
    Converts the number of seconds from unix epoch (1970-01-01 00:00:00 UTC) to a string
    representing the timestamp of that moment in the current system time zone in the given
    format.
    
    .. versionadded:: 1.5.0
    
    .. versionchanged:: 3.4.0
        Supports Spark Connect.
    
    Parameters
    ----------
    timestamp : :class:`~pyspark.sql.Column` or str
        column of unix time values.
    format : str, optional
        format to use to convert to (default: yyyy-MM-dd HH:mm:ss)
    
    Returns
    -------
    :class:`~pyspark.sql.Column`
        formatted timestamp as string.
    
    Examples
    --------
    >>> spark.conf.set("spark.sql.session.timeZone", "America/Los_Angeles")
    >>> time_df = spark.createDataFrame([(1428476400,)], ['unix_time'])
    >>> time_df.select(from_unixtime('u